# CurvLearn — Mechanism comparison: 'place, don't score'

Tree embedding (A″) showed hyperbolic geometry is sound — it halves tree-embedding distortion.
The Dyck LM (A′) showed hyperbolic *hurts* when curvature is put in the **attention scores**
(`-β·d_κ²`). Thesis: the geometry is fine, the mechanism is wrong.

This compares two attention mechanisms on the same difficulty sweep:

- **geodesic** — curvature in the scores (Design A′; expected to hurt at κ<0).
- **gyro** — tangent-space Q·K scores, curvature enters via a **hyperbolic gyromidpoint
  aggregation** of the value points. This is where placement genuinely matters (a tangent
  weighted sum would cancel κ, since log₀∘exp₀ = id).

If **gyro** is flat or dips **below 0** at κ<0 where **geodesic** rises above it, the fix works
and Design B (scheduling κ) finally has a mechanism that can use the curvature.


## 1. GPU


In [ ]:
!nvidia-smi -L || echo 'CPU (slow for ~90 runs)'

## 2. Code + install (after a `git pull`, restart the runtime to pick up changes)


In [ ]:
import os
if not os.path.isdir('CurvLearn') and os.path.basename(os.getcwd()) != 'CurvLearn':
    !git clone https://github.com/TheFausap/CurvLearn.git
if os.path.basename(os.getcwd()) != 'CurvLearn':
    os.chdir('CurvLearn')
!pip -q install -e .
print('cwd:', os.getcwd())

## 3. Sanity: geometry (incl. gyromidpoint) + trees


In [ ]:
!python tests/test_geometry.py
!python tests/test_trees.py

## 4. Drive + output


In [ ]:
import os, torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/CurvLearn/results'
except ModuleNotFoundError:
    OUTDIR = 'results'
os.makedirs(OUTDIR, exist_ok=True)
print('device:', dev, '| results ->', OUTDIR)

## 5. Run: difficulty sweep x {geodesic, gyro}
2 modes x 3 difficulties x 5 κ x 3 seeds = 90 runs, ~1-1.5 h on an A100. Checkpointed to a
NEW file (mechanism_results.json). Drop `seeds` to 2 or a difficulty for a faster first look.


In [ ]:
import json
from curvlearn.difficulty import run_difficulty, default_spec

spec = default_spec()                     # attention_modes = ['geodesic','gyro']
spec['train']['device'] = dev
ckpt = os.path.join(OUTDIR, 'mechanism_results.json')
out = run_difficulty(spec, checkpoint_path=ckpt, resume=True)
print('runs:', len(out['records']), '| saved ->', ckpt)

## 6. Plot: geodesic vs gyro


In [ ]:
import json
from curvlearn import plot
from IPython.display import Image, display
out = json.load(open(os.path.join(OUTDIR, 'mechanism_results.json')))
png = os.path.join(OUTDIR, 'mechanism_compare.png')
plot.mechanism_plot(out, png)
display(Image(png))
print('saved', png)

## 7. Read it
- **gyro dips below 0 at κ<0 (esp. hard) while geodesic rises above** → hyperbolic aggregation
  converts the geometric capacity into a modelling gain. The thesis holds; Design B is on.
- **gyro flat, geodesic rises** → the fix removes the *penalty* but not into a *benefit*;
  next rung is deeper hyperbolic ops (Möbius linear / hyperbolic FFN) or a tree-distance target.
- **gyro also rises at κ<0** → even correct hyperbolic aggregation doesn't help this LM task;
  the hierarchy here may just not be the kind hyperbolic space buys.
